## Set-up

In [1]:
import sys
from pathlib import Path

# >>>> CHANGE THIS if your path is a bit different <<<<
SRC_ROOT = Path("/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine/src")

print("SRC_ROOT exists? ", SRC_ROOT.exists())
print("SRC_ROOT path    ", SRC_ROOT)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("Added to sys.path")


SRC_ROOT exists?  True
SRC_ROOT path     /Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine/src
Added to sys.path


In [2]:
import pkgutil

print("job_intel visible in SRC_ROOT?:",
      any(m.name == "job_intel" for m in pkgutil.iter_modules([str(SRC_ROOT)])))


job_intel visible in SRC_ROOT?: True


In [3]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

## Create data from pipeline

In [4]:
from job_intel.pipelines.chapter0_build_base_dataset import build_chapter0_base_dataset

df_ch0 = build_chapter0_base_dataset()
df_ch0.head()


Loading raw jobs ...
Raw combined shape: (6162, 16)
Replaced -1 with NAs
Added location features.
Simplified ownership.
Converted year to integer.
Dropped uneccesary features (Nas and extremely complex) - Easy Apply, Competitors, Company Name and Revenue.
Added job_description_clean.
Added title/seniority/family features (using cleaned description).
Added domain from lookup (missing=0).
Added salary features from 'Salary Estimate'.
Added skill flag features.
Dropping columns: ['Job Title', 'job_title_for_skills', 'job_title_raw', 'Salary Estimate', 'seniority_roman', 'seniority_title', 'seniority_description', 'state_hq']
Saved Chapter 0 processed dataset to: /Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine/data/processed/jobs_ch0.csv (shape=(6162, 47))


,Job Description,Rating,Size,Founded,Industry,Sector,role_source,state,ownership_clean,job_description_clean,...,cloud__advanced,db_storage__basic,db_storage__intermediate,db_storage__advanced,productivity_workflow__basic,productivity_workflow__intermediate,productivity_workflow__advanced,soft_skills__core,soft_skills__leadership,domain_specific__none
0,"ABOUT HOPPER\n\nAt Hopper, we’re on a mission ...",3.5,501 to 1000 employees,2007,Travel Agencies,Travel & Tourism,data_scientist,NY,private,hopper mission make booking travel faster easi...,...,0,1,0,0,0,0,0,1,0,0
1,"At Noom, we use scientifically proven methods ...",4.5,1001 to 5000 employees,2008,"Health, Beauty, & Fitness",Consumer Services,data_scientist,NY,private,noom use scientifically proven methods help us...,...,0,1,0,0,0,0,0,1,0,0
2,Decode_M\n\nhttps://www.decode-m.com/\n\nData ...,NaN,1 to 50 employees,<NA>,Unknown,Unknown,data_scientist,NY,unknown,decode https www com data science manager job ...,...,0,0,0,0,0,0,0,1,1,1
3,Sapphire Digital seeks a dynamic and driven mi...,3.4,201 to 500 employees,2019,Internet,Information Technology,data_scientist,NJ,private,sapphire digital seeks dynamic driven mid leve...,...,0,1,0,0,0,0,0,1,1,0
4,"Director, Data Science - (200537)\nDescription...",3.4,51 to 200 employees,2007,Advertising & Marketing,Business Services,data_scientist,NY,private,director data science description edelman inte...,...,0,0,0,0,0,0,0,1,1,0


### Benchmark data

In [5]:
from job_intel.config import INTERIM_DATA_DIR
benchmark = pd.read_csv(INTERIM_DATA_DIR / '05_skills_extracted.csv')

## Benchmark test

In [6]:
df_ch0.columns

Index(['Job Description', 'Rating', 'Size', 'Founded', 'Industry', 'Sector',
       'role_source', 'state', 'ownership_clean', 'job_description_clean',
       'job_title_base', 'seniority_combined', 'job_title_norm',
       'job_title_family', 'domain', 'sal_is_hourly', 'sal_min', 'sal_max',
       'sal_mean', 'title_plus_description', 'core_programming__basic',
       'core_programming__intermediate', 'core_programming__advanced',
       'data_engineering_pipelines__basic',
       'data_engineering_pipelines__intermediate',
       'data_engineering_pipelines__advanced', 'ml_ai__basic',
       'ml_ai__intermediate', 'ml_ai__advanced', 'analytics_stats__basic',
       'analytics_stats__intermediate', 'analytics_stats__advanced',
       'bi_viz__basic', 'bi_viz__intermediate', 'bi_viz__advanced',
       'cloud__basic', 'cloud__intermediate', 'cloud__advanced',
       'db_storage__basic', 'db_storage__intermediate', 'db_storage__advanced',
       'productivity_workflow__basic', 'productiv

In [7]:
columns_to_check = [
    "Rating",
    "Size",
    "Founded",
    "Industry",
    "Sector",
    "state",
    "ownership_clean",
    "job_description_clean",
    "job_title_base",
    "seniority_combined",
    "job_title_norm",
    "job_title_family",
    "domain",
    "sal_mean",'core_programming__basic',
    'core_programming__intermediate', 'core_programming__advanced',
    'data_engineering_pipelines__basic',
    'data_engineering_pipelines__intermediate',
    'data_engineering_pipelines__advanced', 'ml_ai__basic',
    'ml_ai__intermediate', 'ml_ai__advanced', 'analytics_stats__basic',
    'analytics_stats__intermediate', 'analytics_stats__advanced',
    'bi_viz__basic', 'bi_viz__intermediate', 'bi_viz__advanced',
    'cloud__basic', 'cloud__intermediate', 'cloud__advanced',
    'db_storage__basic', 'db_storage__intermediate', 'db_storage__advanced',
    'productivity_workflow__basic', 'productivity_workflow__intermediate',
    'productivity_workflow__advanced', 'soft_skills__core',
    'soft_skills__leadership', 'domain_specific__none',
]


for col in columns_to_check:
    # Basic guards
    if col not in df_ch0.columns or col not in benchmark.columns:
        print(f"[{col}] SKIPPED – column missing in one of the DataFrames")
        continue
    if len(df_ch0) != len(benchmark):
        print(f"[{col}] SKIPPED – different lengths: {len(df_ch0)} vs {len(benchmark)}")
        continue

    s1 = df_ch0[col]
    s2 = benchmark[col]

    # Numeric columns → use np.isclose + equal_nan
    if pd.api.types.is_numeric_dtype(s1):
        mask = np.isclose(s1, s2, equal_nan=True)
    else:
        # Non-numeric columns → exact match OR both NaN
        mask = (s1 == s2) | (s1.isna() & s2.isna())

    match_ratio = mask.mean()

    if match_ratio == 1:
        print(f"✅[{col}] OK – perfect match")
    else:
        n_mismatch = (~mask).sum()
        print(f"❌[{col}] FAIL – {match_ratio:.3f} match, {n_mismatch} mismatches")



✅[Rating] OK – perfect match
❌[Size] FAIL – 0.936 match, 392 mismatches
✅[Founded] OK – perfect match
❌[Industry] FAIL – 0.854 match, 899 mismatches
❌[Sector] FAIL – 0.854 match, 899 mismatches
✅[state] OK – perfect match
✅[ownership_clean] OK – perfect match
❌[job_description_clean] FAIL – 1.000 match, 1 mismatches
✅[job_title_base] OK – perfect match
✅[seniority_combined] OK – perfect match
✅[job_title_norm] OK – perfect match
✅[job_title_family] OK – perfect match
✅[domain] OK – perfect match
✅[sal_mean] OK – perfect match
✅[core_programming__basic] OK – perfect match
✅[core_programming__intermediate] OK – perfect match
✅[core_programming__advanced] OK – perfect match
✅[data_engineering_pipelines__basic] OK – perfect match
✅[data_engineering_pipelines__intermediate] OK – perfect match
✅[data_engineering_pipelines__advanced] OK – perfect match
✅[ml_ai__basic] OK – perfect match
✅[ml_ai__intermediate] OK – perfect match
✅[ml_ai__advanced] OK – perfect match
✅[analytics_stats__basic] O

### Size, Industry and Sector check

In [ ]:
for col in ["Size", "Industry", "Sector"]:
    mask = df_ch0[col] != benchmark[col]
    print(f"\n=== {col} mismatches: {mask.sum()} rows ===")

    # Quick sanity: what did the new pipeline assign?
    print("Pipeline values in mismatches:")
    print(df_ch0.loc[mask, col].value_counts().head())

    print("Benchmark values in mismatches:")
    print(benchmark.loc[mask, col].value_counts().head())


This means that the fail is coming from the NA filling within the pipeline generator to Unknown for these columns.

### Seniority check

In [ ]:
tmp = pd.DataFrame({
    "bench_combined": benchmark["seniority_combined"],
    "pipe_combined":  df_ch0["seniority_combined"],
    "title":          benchmark["seniority_title"],        # same as pipeline
    "desc":           benchmark["seniority_description"],  # same as pipeline
})

mism = tmp[tmp["bench_combined"] != tmp["pipe_combined"]]

mism_summary = (
    mism
    .groupby(["title", "desc", "bench_combined", "pipe_combined"])
    .size()
    .sort_values(ascending=False)
)

mism_summary.head(30)



### Job description check

In [ ]:
col = "job_description_clean"

s1 = df_ch0[col]
s2 = benchmark[col]

# True mismatch = values differ AND are not both NaN
mask = (s1 != s2) & ~(s1.isna() & s2.isna())
print("True mismatches:", mask.sum())

mismatch_idx = df_ch0.index[mask]
mismatch_idx


In [ ]:
cols_to_show = [
    "job_title_norm",
    "job_title_base",
    "job_description_clean",
]

print("PIPELINE VERSION:")
display(df_ch0.loc[mismatch_idx, cols_to_show])

print("BENCHMARK VERSION:")
display(benchmark.loc[mismatch_idx, cols_to_show])


The job description issue is just NA vs "". All good

## General describes

In [ ]:
len(df_ch0.columns) # == 47

In [ ]:
df_ch0.describe(include = 'O')

In [ ]:
df_ch0.describe()

In [ ]:
df_ch0.info()

In [ ]:
df_ch0

## Check NAs

In [ ]:
df_ch0.isna().apply(lambda x: (sum(x) / (len(df_ch0)))*100).sort_values().plot(kind = 'bar')
plt.ylabel('% of total data missing')
plt.xlabel('Feature')
plt.tight_layout()

df_ch0.isna().apply(lambda x: (sum(x) / (len(df_ch0)))*100).sort_values(ascending=False)

In [ ]:

df_ch0[df_ch0['sal_mean'].isna()]

## Check seniority

In [ ]:
df_ch0['seniority_title'].value_counts()

## Check job title

In [ ]:
df_ch0['job_title_base'].value_counts()

In [ ]:
df_ch0['job_title_family'].value_counts()

In [ ]:
df_ch0["job_title_clean"] = df_ch0["domain"].astype(str) + " " + df_ch0["job_title_family"]
df_ch0["job_title_clean"].value_counts()

## Check alignment of EDA with notebook 00_

In [ ]:
sns.countplot(
    data = df_ch0,
    x = 'state',
    color='grey',
    edgecolor = 'black',
    order=df_ch0['state'].value_counts().index
)
plt.xticks(rotation = 90)
plt.xlabel('State / Country')
plt.ylabel('Count')
plt.title('Density of job locations')
plt.tight_layout()

In [ ]:
sns.histplot(
    data = df_ch0,
    x = 'sal_mean',
    color='grey',
    edgecolor = 'black',
    bins=50,
    kde=True,
)

plt.xlabel('Mean salary')
plt.ylabel('Count')
plt.title('Average Salary Distribution')
plt.tight_layout()

In [ ]:
order = df_ch0['domain'].value_counts().index

sns.countplot(
    data = df_ch0,
    x = 'domain',
    order = order
)
plt.xticks(rotation = 90)
plt.tight_layout()

In [ ]:
order = df_ch0['seniority_title'].value_counts().index

sns.countplot(
    data = df_ch0,
    x = 'seniority_title',
    order = order
)
plt.xticks(rotation = 90)
plt.tight_layout()

## Check skill extraction

In [ ]:
print(df_ch0.loc[25, "title_plus_description"])

data scientist  marketplace economics looking data scientist join marketplace economics team within business group mission use science shaping efforts improve spotify strengthening creator sustainability accomplishes combining incentive structure creators intermediate agents labels publishers governmental organizations regulatory bodies internal external test empirical hypotheses build algorithmic measurement performance indicators tools services role using petabytes help products surface insights understand connections artists listeners track impact careers centralized researchers scientists novel research addition possessing strong technical background natural communicator equally comfortable explaining complex economic frameworks engineering teams also preference learning applying causal inference econometric machine statistical techniques accompanying broad set responsibilities exposure many functional areas well senior management across building reporting worldwide music industry trends design maintain iterate suite enable metrics measuring health work closely phd level identify content substitution patterns transition self serve projects balance needs two sided considering goals maintaining listener satisfaction discovery communicate driven recommendations key partners clear visualizations presentations degree mathematics statistics similar quantitative field ability topics plain non language prior proven experience analyzing summarizing via modern sql dialects scripting python preferred stata r visualization dashboards tableau working distributed multiple offices timezones communicative person values relationships colleagues explain simple terms welcome matter come look like playing headphones platform everyone workplace voices represented amplified thrive contribute brilliant bring personal perspectives differences find power keep revolutionizing way world listens transformed listening forever launched unlock potential human creativity giving million creative opportunity live art billions fans enjoy inspired everything love podcasting today popular audio streaming subscription service community users

In [ ]:
df_ch0.loc[25, "domain"]

In [ ]:
df_ch0.loc[25, "seniority_combined"]

In [ ]:
from job_intel.features.skill_extractor import explain_matches

row0_text = df_ch0.loc[25, "title_plus_description"]
matches0 = explain_matches(row0_text)
matches0

In [ ]:
df_ch0.loc[25, 'data_engineering_pipelines__intermediate']

In [ ]:
skill_cols = [c for c in df_ch0.columns if "__" in c]

summary = df_ch0[skill_cols].sum().sort_values(ascending=False)
summary = (
    summary
        .reset_index()
        .rename(columns={"index": "skill_domain", 0: "count"})
)

summary

In [ ]:
sns.lineplot(
    data = summary,
    x = 'skill_domain',
    y = 'count'
)
plt.xticks(rotation = 90)
plt.tight_layout()